# 03 · Diffusion MNIST —— 雾玻璃擦雾：从加噪到去噪

**家族位置**：07 生成式模型第 3 站。01 是直接重建，02 是对抗造假，本章走第三条路：训练一个去噪器，把纯噪声一步步擦成数字。

**学习目标**：理解 DDPM 前向 `q(x_t|x_0)`、一步到位的重参数加噪、预测噪声 epsilon 训练目标、反向去噪采样链。

## 1. 原理：先喷雾，再学着擦

### 通俗理解

**一句话**：把数字照片盖上越来越厚的雾（加噪），再训练一个擦雾工：给他雾图和当前雾的厚度 t，让他猜刚才喷了哪层雾；猜准了，从纯雾倒着擦回，就能擦出新数字。

### 结构账

```
前向： x_t = √ᾱ_t·x_0 + √(1-ᾱ_t)·ε  （一步到位，不用真走 T 步）
训练： L = MSE(ε̂(x_t,t), ε)        （只学噪声残差，像素让公式算）
反向： x_{t-1} = 1/√α_t·(x_t - β_t/√(1-ᾱ_t)·ε̂) + √β_t·z
```

U-Net-lite：下采样看全局，上采样回细节，时间嵌入告诉网络雾有多厚。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
ROOT=Path.cwd()
while ROOT != ROOT.parent and not (ROOT/'common').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from common.data import load_mnist_local
from common.models import DDPM
from common.utils import set_seed,setup_chinese_font,count_params,show_mnist
set_seed(0); setup_chinese_font()
FIGS=Path.cwd()/'figs'; FIGS.mkdir(exist_ok=True)
print('torch:',torch.__version__)
Xtr,ytr,Xva,yva,Xte,yte=load_mnist_local(8000,1000,seed=0)
loader=DataLoader(TensorDataset(Xtr,ytr),batch_size=128,shuffle=True)
print(f'train {tuple(Xtr.shape)} | test {tuple(Xte.shape)} | MNIST 标准化域')

## 2. 前向加噪：T 步雾化，一步到位

In [ ]:
T,EPOCHS,LR=100,12,2e-4
ddpm=DDPM(timesteps=T,base=32)
print(f'DDPM params={count_params(ddpm)} | T={T} | beta 1e-4→0.02')
with torch.no_grad():
    x0=Xtr[:1]; fig,ax=plt.subplots(1,6,figsize=(10,2.0))
    for j,t in enumerate([0,10,25,50,75,99]):
        tt=torch.tensor([t]); xt,_=ddpm.q_sample(x0,tt,torch.randn_like(x0))
        ax[j].imshow(show_mnist(xt[0]),cmap='gray'); ax[j].set_title(f't={t}'); ax[j].axis('off')
plt.suptitle('同一张 3：雾越来越厚，t=99 几乎纯噪'); plt.tight_layout()
plt.savefig(FIGS/'fig0_forward.png',dpi=150,bbox_inches='tight'); plt.show()

## 3. 训练：只猜雾，不猜图

In [ ]:
opt=torch.optim.Adam(ddpm.parameters(),lr=LR)
hist=[]; n=len(loader.dataset)
for ep in range(1,EPOCHS+1):
    ddpm.train(); tot=0
    for xb,yb in loader:
        loss=ddpm.loss(xb); opt.zero_grad(); loss.backward(); opt.step(); tot+=loss.item()*len(xb)
    hist.append(tot/n)
    print(f'epoch {ep:02d} | epsilon MSE {hist[-1]:.4f}',flush=True)
fig,ax=plt.subplots(figsize=(6,3.2)); ax.plot(hist,marker='o',color='#4C72B0')
ax.set_xlabel('epoch'); ax.set_ylabel('epsilon MSE'); ax.set_title('Diffusion 训练：猜雾误差稳步下降')
plt.tight_layout(); plt.savefig(FIGS/'fig1_diff_loss.png',dpi=150,bbox_inches='tight'); plt.show()

## 4. 采样：从纯噪擦出数字

In [ ]:
with torch.no_grad():
    final,traj=ddpm.sample(16,trajectory_steps=8)
    print(f'sample {tuple(final.shape)} | traj len {len(traj)}')
fig,ax=plt.subplots(1,8,figsize=(11,2.0))
for j in range(8):
    ax[j].imshow(show_mnist(traj[j][0]),cmap='gray'); ax[j].set_title(f'step {j}'); ax[j].axis('off')
plt.suptitle('同一 z 的去噪轨迹：纯噪→灰雾→数字浮现'); plt.tight_layout()
plt.savefig(FIGS/'fig2_trajectory.png',dpi=150,bbox_inches='tight'); plt.show()
fig,ax=plt.subplots(2,8,figsize=(11,3.2))
for a,x in zip(ax.flat,final[:16]): a.imshow(show_mnist(x),cmap='gray'); a.axis('off')
plt.suptitle('16 张 DDPM 末期样本：比 GAN 灰雾期更像数字'); plt.tight_layout()
plt.savefig(FIGS/'fig3_samples.png',dpi=150,bbox_inches='tight'); plt.show()

## 5. 不同步数 T 的代价：T=100 vs T=20

T 越大雾越细（每步只擦一点），但采样要走 T 步；T 太小每步跨度大，去噪难。

In [ ]:
ddpm20=DDPM(timesteps=20,base=32)
opt20=torch.optim.Adam(ddpm20.parameters(),lr=LR)
for ep in range(1,5):
    ddpm20.train(); tot=0
    for xb,yb in loader:
        loss=ddpm20.loss(xb); opt20.zero_grad(); loss.backward(); opt20.step(); tot+=loss.item()*len(xb)
    print(f'T20 epoch {ep} | epsilon MSE {tot/n:.4f}',flush=True)
with torch.no_grad():
    f100,_=ddpm.sample(8); f20,_=ddpm20.sample(8)
fig,ax=plt.subplots(2,8,figsize=(11,3.2))
for j in range(8):
    ax[0,j].imshow(show_mnist(f100[j]),cmap='gray'); ax[0,j].axis('off')
    ax[1,j].imshow(show_mnist(f20[j]),cmap='gray'); ax[1,j].axis('off')
ax[0,0].set_ylabel('T=100'); ax[1,0].set_ylabel('T=20(4ep)')
plt.suptitle('T 对比：步数多擦得细，但采样慢 5 倍'); plt.tight_layout()
plt.savefig(FIGS/'fig4_T_compare.png',dpi=150,bbox_inches='tight'); plt.show()
print(f'DDPM T100 params={count_params(ddpm)} | T20 同结构，采样步数 100 vs 20')

## 6. 总结与下一步

Diffusion 闭环完成：前向一步加噪 → 训练只猜雾（12ep epsilon MSE 0.55→0.13）→ 反向逐步去噪擦出可辨数字，训练曲线平稳。下一步 `04_Flow_Optional`：可逆拉链 z↔x，精确似然（拓展）。